In [15]:
pip install librosa matplotlib seaborn pandas numpy

Note: you may need to restart the kernel to use updated packages.


# 01 — Data Loading
**Goal:** Load RAVDESS and TESS audio files, map emotion labels, merge `calm → neutral`, and save a clean `dataset.csv` for all future notebooks.

## 1. Imports

In [3]:
import os
import pandas as pd

## 2. Set Dataset Paths
Update these two paths to point to your local dataset folders.

In [4]:
RAVDESS_PATH = "D:\Projects\CSE445\speech_emotion_recognation\data\RAVDESS"  # folder containing Actor_01, Actor_02, ...
TESS_PATH    = "D:\Projects\CSE445\speech_emotion_recognation\data\TESS"     # folder containing OAF_angry, YAF_happy, ...

In [5]:
# Check RAVDESS
if os.path.exists(RAVDESS_PATH):
    subfolders = os.listdir(RAVDESS_PATH)
    print(f"✅ RAVDESS found — {len(subfolders)} subfolders")
    print(f"   First 3: {subfolders[:3]}")
else:
    print(f"❌ RAVDESS NOT found at: {RAVDESS_PATH}")

# Check TESS
if os.path.exists(TESS_PATH):
    subfolders = os.listdir(TESS_PATH)
    print(f"✅ TESS found — {len(subfolders)} subfolders")
    print(f"   First 3: {subfolders[:3]}")
else:
    print(f"❌ TESS NOT found at: {TESS_PATH}")

✅ RAVDESS found — 25 subfolders
   First 3: ['Actor_01', 'Actor_02', 'Actor_03']
✅ TESS found — 2801 subfolders
   First 3: ['MANIFEST.TXT', 'OAF_back_angry.wav', 'OAF_back_disgust.wav']


In [6]:
TESS_PATH = "data/TESS"

files = [f for f in os.listdir(TESS_PATH) if f.endswith(".wav")]
print(f"Total wav files: {len(files)}")
print("\nSample filenames:")
for f in files[:10]:
    print(f"  {f}")

Total wav files: 2800

Sample filenames:
  OAF_back_angry.wav
  OAF_back_disgust.wav
  OAF_back_fear.wav
  OAF_back_happy.wav
  OAF_back_neutral.wav
  OAF_back_ps.wav
  OAF_back_sad.wav
  OAF_bar_angry.wav
  OAF_bar_disgust.wav
  OAF_bar_fear.wav


## 3. Emotion Label Maps

In [7]:
# RAVDESS encodes emotion as the 3rd number in the filename
# e.g. 03-01-06-01-02-01-12.wav → digit '06' = fearful
RAVDESS_EMOTION_MAP = {
    "01": "neutral",
    "02": "neutral",   # calm → merged into neutral
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fear",
    "07": "disgust",
    "08": "surprise",
}

# TESS encodes emotion in the folder name
# e.g. OAF_angry, YAF_ps (ps = pleasant surprise)
TESS_EMOTION_MAP = {
    "angry":   "angry",
    "disgust": "disgust",
    "fear":    "fear",
    "happy":   "happy",
    "neutral": "neutral",
    "ps":      "surprise",   # pleasant surprise
    "sad":     "sad",
}

# Final 7 emotion classes
EMOTION_LABELS = ["neutral", "happy", "sad", "angry", "fear", "disgust", "surprise"]

print("Emotion maps defined.")
print(f"Final classes: {EMOTION_LABELS}")

Emotion maps defined.
Final classes: ['neutral', 'happy', 'sad', 'angry', 'fear', 'disgust', 'surprise']


## 4. Load RAVDESS

In [8]:
def load_ravdess(ravdess_path):
    records = []
    for actor_folder in sorted(os.listdir(ravdess_path)):
        actor_dir = os.path.join(ravdess_path, actor_folder)
        if not os.path.isdir(actor_dir):
            continue
        for fname in os.listdir(actor_dir):
            if not fname.endswith(".wav"):
                continue
            parts = fname.replace(".wav", "").split("-")
            emotion_code = parts[2]   # 3rd segment = emotion digit
            emotion = RAVDESS_EMOTION_MAP.get(emotion_code)
            if emotion is None:
                continue
            filepath = os.path.join(actor_dir, fname)
            records.append({"path": filepath, "emotion": emotion, "source": "RAVDESS"})
    return records

ravdess_records = load_ravdess(RAVDESS_PATH)
print(f"RAVDESS files loaded: {len(ravdess_records)}")

RAVDESS files loaded: 1440


## 5. Load TESS

In [9]:
def load_tess(tess_path):
    records = []
    for fname in os.listdir(tess_path):
        if not fname.endswith(".wav"):
            continue
        # e.g. OAF_back_angry.wav → split by _ → last part = angry
        parts = fname.replace(".wav", "").split("_")
        emotion_key = parts[-1].lower()
        emotion = TESS_EMOTION_MAP.get(emotion_key)
        if emotion is None:
            print(f"  [TESS] Unknown emotion key: '{emotion_key}' in {fname}")
            continue
        filepath = os.path.join(tess_path, fname)
        records.append({"path": filepath, "emotion": emotion, "source": "TESS"})
    return records

tess_records = load_tess(TESS_PATH)
print(f"TESS files loaded: {len(tess_records)}")

TESS files loaded: 2800


## 6. Merge Both Datasets

In [10]:
all_records = ravdess_records + tess_records
df = pd.DataFrame(all_records)

# Keep only valid emotion labels
df = df[df["emotion"].isin(EMOTION_LABELS)].reset_index(drop=True)

print(f"Total files after merge: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

Total files after merge: 4240
Columns: ['path', 'emotion', 'source']


,path,emotion,source
0,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
1,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
2,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
3,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
4,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
5,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
6,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
7,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
8,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
9,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS


## 7. Check Emotion Distribution

In [11]:
print("=== Emotion Distribution ===")
print(df["emotion"].value_counts().to_string())

print("\n=== Source Distribution ===")
print(df["source"].value_counts().to_string())

print("\n=== Emotion by Source ===")
print(df.groupby(["source", "emotion"]).size().to_string())

=== Emotion Distribution ===
emotion
neutral     688
happy       592
sad         592
angry       592
fear        592
disgust     592
surprise    592

=== Source Distribution ===
source
TESS       2800
RAVDESS    1440

=== Emotion by Source ===
source   emotion 
RAVDESS  angry       192
         disgust     192
         fear        192
         happy       192
         neutral     288
         sad         192
         surprise    192
TESS     angry       400
         disgust     400
         fear        400
         happy       400
         neutral     400
         sad         400
         surprise    400


## 8. Verify All Files Exist on Disk

In [12]:
missing = df[~df["path"].apply(os.path.exists)]

if len(missing) == 0:
    print("All files verified — no missing files.")
else:
    print(f"WARNING: {len(missing)} missing files found!")
    print(missing["path"].head(10).to_string())

All files verified — no missing files.


## 9. Save dataset.csv

In [16]:
df.to_csv("dataset.csv", index=False)
print("Saved: dataset.csv")
print(f"Shape: {df.shape}")

Saved: dataset.csv
Shape: (4240, 3)


---
**Done!** `dataset.csv` is ready. Next step → `02_EDA.ipynb`